## 3.2 — Creating arrays, dtypes, and inspecting shape
 
### 1. Creating arrays — common constructors
 
**From existing data:**

In [1]:
import numpy as np
np.array([[0, 3, 1], [2, 0, 5]])     # list-of-lists -> 2-D array

array([[0, 3, 1],
       [2, 0, 5]])

**Generated from scratch** (when you need a shape before you have values, or a placeholder to fill):

In [2]:
np.zeros((3, 4))        # 3x4 of 0.0  — blank matrix to fill later
np.ones((2, 3))         # 2x3 of 1.0
np.full((2, 2), 7)      # 2x2, every value 7
np.arange(0, 10, 2)     # [0 2 4 6 8]  — like range(), returns an array
np.linspace(0, 1, 5)    # [0. 0.25 0.5 0.75 1.] — 5 evenly spaced points, endpoints included

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

**`arange` takes a *step*; `linspace` takes a *count*.** `arange(0,1,0.1)` can miss the endpoint from float rounding; `linspace(0,1,11)` guarantees exactly 11 inclusive points. For a guaranteed number of evenly-spaced values, prefer `linspace`.
 
Relevance: `np.zeros` preallocates a results matrix before a computation fills it (right pattern vs. growing in a loop); `arange`/`linspace` build axis values, thresholds, histogram bins.
 
### 2. dtype — the type contract, and why it bites in scRNA-seq
 
Every array has one `dtype`, which you can set explicitly.

In [3]:
counts = np.array([[0, 3, 1], [2, 0, 5]])                     # dtype('int64')
expr   = np.array([[0, 3, 1], [2, 0, 5]], dtype=np.float32)   # forced float32


**(a) Counts are integers; expression is float.** Raw UMI counts are whole numbers → int. Normalizing/log-transforming makes values fractional → float. Scanpy stores expression as float (usually `float32`).
 
**(b) `float32` vs `float64` is a real memory decision.** `float64` = 8 bytes/value (NumPy's float default); `float32` = 4 bytes. On 50,000 × 20,000 that's ~8 GB vs ~4 GB. Scanpy defaults to `float32` to halve memory, trading slight precision — correct for this data. **Memory is the binding constraint in single-cell work, and dtype is a lever on it.**
 
**Silent truncation trap:**

In [4]:
a = np.array([1, 2, 3])    # int64
a[0] = 3.9                 # float assigned into an int array...
a[0]

np.int64(3)

The array doesn't change type to fit your float; it forces the float into an int and drops the fraction.

### 3. Inspecting an array you didn't create — the reflexive first move
 
When a matrix arrives (file, scanpy, colleague), inspect *before* computing:

In [10]:
X = np.zeros((2000, 15000), dtype=np.float32)   # pretend: a loaded dataset
 
print(X.shape)    # (2000, 15000) -> 2000 cells, 15000 genes (CHECK first, every time)
print(X.ndim)     # 2
print(X.size)     # 30_000_000 total entries
print(X.dtype)    # float32 -> already normalized (floats, not counts)
print(X.nbytes)   # 120_000_000 bytes (~120 MB) -> memory footprint

(2000, 15000)
2
30000000
float32
120000000


The in-head reasoning:
- `shape` → "2000 cells, 15000 genes — does 2000 match my barcodes? If 15000 were first, suspect a transpose."
- `dtype` → "float32 → not raw counts; someone normalized. Where's raw?"
- `nbytes` → "scaled to my real matrix, will it fit in RAM?"
### 4. Common mistakes
 
- **Growing arrays in a loop** (`np.append` in a `for`) — each append reallocates and copies the whole array → O(n²). Preallocate with `np.zeros(final_shape)` and fill; if size is unknown, collect in a Python list and convert once.
- **Ignoring the default float dtype** — `np.zeros((n,m))` is `float64`; pass `dtype=np.float32` for large matrices.
- **Assuming int arrays hold floats** — assignment coerces to the array dtype silently (truncation trap).
- **Not inspecting before computing** — discovering a transpose or stale int dtype three steps later.
### 5. Best practices
 
- **Preallocate, don't grow.**
- **Set dtype explicitly for large numeric data** — `float32` for expression unless there's a reason for `float64`.
- **Inspect on arrival:** `.shape`, `.dtype`, `.nbytes` reflexively.
- **Read `.shape` as `(n_cells, n_genes)`** and sanity-check against known counts.
### 6. Related constructors
 
- **`np.empty(shape)`** — like `zeros` but uninitialized (contains memory garbage); only when you'll overwrite every element immediately.
- **`np.zeros_like(other)` / `np.ones_like(other)`** — new array matching another's shape *and* dtype.

### 7. Hands-on
 

In [11]:
import numpy as np
 
# 1. Blank float32 matrix the size of a small dataset
X = np.zeros((500, 2000), dtype=np.float32)   # 500 cells x 2000 genes
 
# 2. Inspect as you would real loaded data
print("shape :", X.shape)      # (500, 2000)
print("dtype :", X.dtype)      # float32
print("nbytes:", X.nbytes)     # 4,000,000 (= 500*2000*4)
 
# 3. dtype trap
counts = np.array([5, 10, 15])  # int64
counts[1] = 7.8
print(counts)                   # [ 5  7 15] -> 7.8 truncated
 
# 4. arange vs linspace
print(np.arange(0, 1, 0.3))     # [0.  0.3 0.6 0.9] -> step-based, endpoint missed
print(np.linspace(0, 1, 4))     # [0. 0.333 0.667 1.] -> count-based, endpoint included


shape : (500, 2000)
dtype : float32
nbytes: 4000000
[ 5  7 15]
[0.  0.3 0.6 0.9]
[0.         0.33333333 0.66666667 1.        ]


`X.nbytes` == `500*2000*4` (4 bytes per `float32`) — the memory model as arithmetic. Switch to `float64` and `nbytes` doubles; that is why scanpy picks `float32`.

### 8. Summary
 
Arrays come from converting data (`np.array`) or generating it (`zeros`, `ones`, `full`, `arange`, `linspace`); `arange` is step-based, `linspace` count-based and safer for a fixed number of points. Each array has one `dtype`: counts are int, normalized expression is float, and `float32` vs `float64` is a 4-vs-8-byte memory decision scanpy settles as `float32`. Assigning a float into an int array truncates silently. Reflexively inspect `.shape` (cells × genes?), `.dtype` (counts or normalized?), `.nbytes` (fits in RAM?) on any matrix you didn't build.
